In [1]:
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

In [2]:
TRAINING_JSON = Path('./../data/raw/training.json')
TEST_JSON = Path('./../data/raw/test.json')
IMAGES_DIR = Path('./../data/raw/images')

In [3]:
with open(TRAINING_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

print(len(data))

1208


In [4]:
item = data[0]

print(json.dumps(item, indent=4, ensure_ascii=False))

{
    "image": {
        "checksum": "676bb8e86fc2dbf05dd97d51a64ac0af",
        "pathname": "/images/8d02117d-6c71-4e47-b50a-6cc8d5eb1d55.png",
        "shape": {
            "r": 1200,
            "c": 1600,
            "channels": 3
        }
    },
    "objects": [
        {
            "bounding_box": {
                "minimum": {
                    "r": 1057,
                    "c": 1440
                },
                "maximum": {
                    "r": 1158,
                    "c": 1540
                }
            },
            "category": "red blood cell"
        },
        {
            "bounding_box": {
                "minimum": {
                    "r": 868,
                    "c": 1303
                },
                "maximum": {
                    "r": 971,
                    "c": 1403
                }
            },
            "category": "red blood cell"
        },
        {
            "bounding_box": {
                "minimum": {
               

In [5]:
# ---------------------------------------------------------
# 1. Load annotations in the REAL JSON format
# ---------------------------------------------------------

def load_detection_annotations(json_path):
    """
    Reads the JSON in the provided format (image + objects + bounding boxes)
    and returns a structured list for later use.
    
    Returned structure:
    [
        {
            'image': 'file_name.png',
            'width': int,
            'height': int,
            'objects': [
                {
                    'bbox': [xmin, ymin, xmax, ymax],
                    'category': string
                },
                ...
            ]
        },
        ...
    ]
    """

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    samples = []

    for item in data:

        raw_path = item["image"]["pathname"]
        fname = Path(raw_path).name

        width = item["image"]["shape"]["c"]
        height = item["image"]["shape"]["r"]

        objs = []
        for obj in item["objects"]:

            bb = obj["bounding_box"]
            ymin = bb["minimum"]["r"]
            xmin = bb["minimum"]["c"]
            ymax = bb["maximum"]["r"]
            xmax = bb["maximum"]["c"]

            objs.append({
                "bbox": [xmin, ymin, xmax, ymax],   # format xmin, ymin, xmax, ymax
                "category": obj["category"]
            })

        samples.append({
            "image": fname,
            "width": width,
            "height": height,
            "objects": objs
        })

    return samples


# ---------------------------------------------------------
# 2. Load image + RGB + numpy
# ---------------------------------------------------------
def load_image(path, as_rgb=True, to_numpy=True):
    img = Image.open(path)

    if as_rgb:
        img = img.convert("RGB")

    if to_numpy:
        return np.array(img)

    return img


# ---------------------------------------------------------
# 3. Build dataset: images + annotations + bounding boxes
# ---------------------------------------------------------
def build_dataset(json_path, images_dir, max_items=None):
    """
    Returns a list of tuples:
    (image_numpy, annotations, real_image_path)

    where annotations =
    {
        'objects': [
            {'bbox': [...], 'category': 'trophozoite'},
            ...
        ],
        'width': W,
        'height': H
    }
    """
    annotations = load_detection_annotations(json_path)
    dataset = []
    missing = []

    for i, entry in enumerate(annotations):

        if max_items and i >= max_items:
            break

        fname = entry["image"]
        img_path = images_dir / fname

        # If not found, try basename only
        if not img_path.exists():
            alt = images_dir / Path(fname).name
            if alt.exists():
                img_path = alt
            else:
                missing.append(fname)
                continue

        try:
            img_arr = load_image(img_path)
        except Exception as e:
            print(f"Error opening {img_path}: {e}")
            continue

        dataset.append((img_arr, entry, str(img_path)))

    print(f"Total read: {len(dataset)}, missing: {len(missing)}")

    if missing:
        print("Missing files (examples):", missing[:10])

    return dataset

In [6]:
dataset = build_dataset(TRAINING_JSON, IMAGES_DIR)

Total read: 1208, missing: 0


In [7]:
dataset[0][0]

array([[[ 85, 101, 122],
        [ 85, 101, 122],
        [ 87, 103, 125],
        ...,
        [238, 240, 239],
        [241, 243, 242],
        [244, 246, 245]],

       [[ 85, 100, 122],
        [ 88, 103, 125],
        [ 90, 105, 127],
        ...,
        [238, 240, 239],
        [241, 243, 242],
        [244, 246, 245]],

       [[ 86, 100, 122],
        [ 88, 103, 125],
        [ 93, 108, 130],
        ...,
        [238, 240, 239],
        [241, 243, 242],
        [244, 246, 245]],

       ...,

       [[ 52,  66,  98],
        [ 55,  69, 102],
        [ 58,  72, 104],
        ...,
        [ 74,  78, 113],
        [ 74,  77, 110],
        [ 76,  79, 113]],

       [[ 47,  62,  97],
        [ 51,  65, 101],
        [ 53,  67, 102],
        ...,
        [ 75,  81, 115],
        [ 78,  82, 115],
        [ 80,  84, 118]],

       [[ 40,  55,  90],
        [ 43,  57,  94],
        [ 46,  60,  96],
        ...,
        [ 75,  81, 115],
        [ 79,  84, 118],
        [ 81,  86, 121]]

In [8]:
# list all images

# for i, (img, label, fname) in enumerate(dataset):
    
#     print(f"Image {i}: {fname} (label = {label})")

#     plt.imshow(img)
#     plt.axis("off")
#     plt.show()